In [ ]:
# %% [markdown]
# # Trajectory and Entropy Visualization (Multi-Mode)
# This notebook compares drone trajectories and entropy trends across multiple planning strategies.

# %%
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_rgba
from matplotlib.lines import Line2D
import seaborn as sns
from PIL import Image
import matplotlib.image as mpimg
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.ticker import MultipleLocator
# New imports for the custom legend handler and GIF generation
import matplotlib.patches as mpatches
from matplotlib.legend_handler import HandlerBase
import imageio
from matplotlib.collections import LineCollection


import io

# %% [markdown]
# ## Load logo and set color palettes

# %%
try:
    logo_arr = mpimg.imread("/home/pantheon/drea/neural_mpc/semantic_mpc/artifacts/results/sun_orientation.png")
except FileNotFoundError:
    print("Logo file not found, creating a placeholder.")
    logo_arr = np.zeros((10, 10, 4)) # Create a transparent placeholder if logo is missing

paired_palette = sns.color_palette("Paired")
dark_paired_palette = sns.color_palette("dark")

# %%
# Define the custom handler for the gradient legend
class HandlerGradient(HandlerBase):
    def __init__(self, cmap, num_stripes=64, **kw):
        super().__init__(**kw)
        self.cmap = cmap
        self.num_stripes = num_stripes

    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        # Create a strip of colors
        gradient = np.linspace(0, 1, self.num_stripes)
        colors = self.cmap(gradient)
        
        # Create a collection of patches (rectangles) for the gradient bar
        stripes = []
        for i in range(self.num_stripes):
            stripe = mpatches.Rectangle([xdescent + i * width / self.num_stripes, ydescent],
                                     width / self.num_stripes,
                                     height,
                                     fc=colors[i],
                                     transform=trans,
                                     edgecolor='none')
            stripes.append(stripe)
        
        return stripes

# Define the custom handler for an image in the legend
class HandlerImage(HandlerBase):
    def __init__(self, image_arr, zoom=1.0, **kw):
        super().__init__(**kw)
        self.image_arr = image_arr
        self.zoom = zoom

    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        # Create an OffsetImage from the image array
        imagebox = OffsetImage(self.image_arr, zoom=self.zoom)
        # Create an AnnotationBbox to place the image at the center of the legend handle's area
        ab = AnnotationBbox(imagebox,
                            (xdescent + 0.5 * width, ydescent + 0.5 * height),
                            frameon=False,
                            xycoords=trans, # Use the transform provided by the legend
                            box_alignment=(0.5, 0.5), # Align the center of the image with the point
                            pad=0)
        return [ab]


# %%
def get_latest_csv(mode, directory, suffix="plot_data.csv"):
    pattern = os.path.join(directory, f"{mode}*{suffix}")
    files = glob.glob(pattern)
    if not files:
        return None
    return max(files, key=os.path.getmtime)

# %%
def load_csv_data(csv_path):
    with open(csv_path, 'r') as f:
        first_line = f.readline().strip()
    parts = first_line.split(',')
    if parts[0] == "tree_positions":
        tree_positions_list = [float(x) for x in parts[1:] if x]
        if tree_positions_list:
            num_trees = len(tree_positions_list) // 2
            tree_positions = np.array(tree_positions_list).reshape(num_trees, 2)
        else:
            tree_positions = np.array([]).reshape(0,2)
    else:
        tree_positions = None

    df = pd.read_csv(csv_path, skiprows=1)
    return (
        df["time"].values,
        df["x"].values,
        df["y"].values,
        df["theta"].values,
        df["entropy"].values,
        df[[col for col in df.columns if col.startswith("lambda_")]].values,
        tree_positions
    )

# %%
def plot_trajectory_subplot(ax, x, y, theta, trees, lambda_history, custom_cmap,
                            tree_label_fontsize, legend_fontsize_param,
                            axis_label_fontsize, tick_label_fontsize,
                            subplot_title_fontsize,
                            mode_label_for_title):
    ax.plot(x, y, marker='o', color='orange', linewidth=2.5, markersize=1.5, label="Drone Trajectory")
    ax.scatter(x[0], y[0], color='crimson', s=250, marker='X', label="Initial Position", zorder=4)
    ax.scatter(x[-1], y[-1], color='gold', s=250, marker='*', label="Final Position", zorder=4)

    if trees is not None and trees.size > 0:
        # Note: The colormap goes from green (low lambda) to red (high lambda)
        for i in range(trees.shape[0]):
            final_lambda = lambda_history[-1, i] if lambda_history.shape[1] > i else 0.5
            tree_color = custom_cmap(final_lambda)
            ax.scatter(trees[i, 0], trees[i, 1], color=tree_color, s=100, marker='o', zorder=3)

    step = max(1, len(x) // 300)
    for idx_arr in range(0, len(x), step):
        x0, y0, t = x[idx_arr], y[idx_arr], theta[idx_arr]
        x1 = x0 + 1.5 * np.cos(t)
        y1 = y0 + 1.5 * np.sin(t)
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="->", color="orange", linewidth=1.5))

    ax.tick_params(axis='both', labelsize=tick_label_fontsize)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-2.0, 18.5)
    ax.set_ylim(-20.5, +1.5)
    ax.set_title(mode_label_for_title, fontsize=subplot_title_fontsize)
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
# %%
def plot_entropy_subplot(ax, time_history, entropy, label, color, axis_label_fontsize, tick_label_fontsize):
    ax.plot(time_history, entropy, linewidth=1.5, markersize=1.0, label=label, color=color)
    ax.set_xlabel("Time (s)", fontsize=axis_label_fontsize)
    ax.set_ylabel("Entropy", fontsize=axis_label_fontsize)
    ax.tick_params(axis='both', labelsize=tick_label_fontsize)
    ax.set_xlim(0, time_history[-1])

# %% [markdown]
# ## Multi-mode Comparison

# %%
modes = ["mpc"] #"greedy", "linear","mower_good", "mower_bad"]
#modes = ["mpc", "greedy", "linear"]
baselines_dir = "decay_test_9trees/run_41" #../to_plot"

axis_label_fontsize = 14
tick_label_fontsize = 14
subplot_title_fontsize = 20
legend_fontsize = 14
suptitle_fontsize = 16
tree_label_fontsize = 10

# The colormap represents the belief: Raw (Green, lambda=0) -> Uncertain (Gray) -> Ripe (Red, lambda=1)
custom_cmap = LinearSegmentedColormap.from_list(
    "custom_cmap", [paired_palette[3], (0.5, 0.5, 0.5), dark_paired_palette[1]]
)
algorithm_labels = {
    "linear": "Linear Path",
    "greedy": "Greedy Approach",
    "mower": "Mower Path",
    "mower_good": "Mower Path - Good View",
    "mower_bad": "Mower Path - Poor View",
    "mpc": "",
}

# %%
# Define grid layout for trajectory plots
nrows = 1
ncols = (len(modes) + nrows - 1) // nrows  # Calculate columns needed, ceiling division

fig_traj, axs_traj = plt.subplots(nrows, ncols, figsize=(6 * 1, 6.5 * nrows), squeeze=False)
axs_flat = axs_traj.flatten() # Flatten the 2D array of axes for easy iteration
fig_entropy, ax_entropy = plt.subplots(figsize=(7.5,3.5))

colors = sns.color_palette("tab10", len(modes))
any_trees = False

# This dictionary will cache the loaded data so we can reuse it for the GIF
data_cache = {}

for i, mode in enumerate(modes):
    label = algorithm_labels.get(mode, mode.capitalize())
    csv_file = get_latest_csv(mode, baselines_dir)
    if not csv_file:
        print(f"No CSV found for {mode}")
        # Turn off the axis for this missing mode
        if i < len(axs_flat):
            axs_flat[i].axis('off')
        continue
    try:
        time_h, x, y, theta, entropy, lambda_h, trees = load_csv_data(csv_file)
        # Cache the data for later use
        data_cache[mode] = {
            "time": time_h, "x": x, "y": y, "theta": theta,
            "entropy": entropy, "lambda_h": lambda_h, "trees": trees
        }
        if trees is not None and trees.size > 0:
            any_trees = True
    except Exception as e:
        print(f"Error loading {csv_file}: {e}")
        # Turn off the axis for this problematic mode
        if i < len(axs_flat):
            axs_flat[i].axis('off')
        continue

    ax_t = axs_flat[i] # Use the flattened array to get the current subplot axis
    plot_trajectory_subplot(ax_t, x, y, theta, trees, lambda_h, custom_cmap,
                            tree_label_fontsize, legend_fontsize,
                            axis_label_fontsize, tick_label_fontsize,
                            subplot_title_fontsize, label)
    oim = OffsetImage(logo_arr, zoom=0.075)
    ab = AnnotationBbox(oim, (-0.01, 1.01), xycoords="axes fraction", frameon=False)
    ax_t.add_artist(ab)

    plot_entropy_subplot(ax_entropy, time_h, entropy, label, colors[i],
                         axis_label_fontsize, tick_label_fontsize)
    
    # *** NEW LINE ADDED HERE ***
    # Add a vertical line at the end time of the run
    #ax_entropy.axvline(x=time_h[-1], color=colors[i], linestyle='--', linewidth=1.5)


# Hide any unused subplots if the number of modes is not a multiple of ncols
for i in range(len(modes), len(axs_flat)):
    axs_flat[i].axis('off')

# %%
# Finalize Trajectory Plot
legend_handles = [
    Line2D([0], [0], color='orange', lw=1.5, marker='o', markersize=2, label='Drone Trajectory'),
    Line2D([0], [0], marker='X', color='w', markerfacecolor='crimson', markersize=15, label='Initial Position'),
    Line2D([0], [0], marker='*', color='w', markerfacecolor='gold', markersize=25, label='Final Position')
]
legend_labels = ['Drone Trajectory', 'Initial Position', 'Final Position']

# Create dummy handles for custom legend items
image_handle = mpatches.Patch(color='none')
legend_handles.append(image_handle)
legend_labels.append('Sun Direction')

# Build the handler map for custom legend artists
handler_map = {
    image_handle: HandlerImage(logo_arr, zoom=0.04) # Adjust zoom for legend size
}

# Conditionally add the gradient legend for tree belief
if any_trees:
    gradient_handle = mpatches.Patch(color='none')
    legend_handles.append(gradient_handle)
    legend_labels.append('Raw(Green)-Ripe(Red) Belief')
    handler_map[gradient_handle] = HandlerGradient(cmap=custom_cmap)

#fig_traj.suptitle("Algorithm Trajectories Comparison", fontsize=suptitle_fontsize)
# Adjust layout to make room for suptitle and legend
fig_traj.tight_layout(rect=[0, 0.05, 1, 0.95])
fig_traj.legend(
    handles=legend_handles, 
    labels=legend_labels,
    loc='lower center', 
    bbox_to_anchor=(0.5, -0.1), 
    ncol=5, 
    fontsize=legend_fontsize,
    # Map the dummy handles to our custom artist handlers
    handler_map=handler_map
)

output_path_traj = os.path.join(baselines_dir, f"decay_thesis_mpc_trajectory_grid.png")
fig_traj.savefig(output_path_traj, format='png', bbox_inches='tight',  dpi=350)

# Finalize Entropy Plot
ax_entropy.set_title("Neural IPP MPC: Continuous Monitoring", fontsize=suptitle_fontsize)
#ax_entropy.legend(loc='upper right',  bbox_to_anchor=(1.4, 1.0))
ax_entropy.grid(True, linestyle='--', alpha=0.9)
output_path_entropy = os.path.join(baselines_dir, f"decay_entropy_comparison.png")
fig_entropy.savefig(output_path_entropy, format='png', bbox_inches='tight', dpi=350)


# %% [markdown]
# ## Individual Tree Entropy Decay Plot

# %%
decay_trends_dir = os.path.join(baselines_dir, "bayes_trends")
decay_files_pattern = os.path.join(decay_trends_dir, "bayes_trend_tree_*.csv")
decay_files = sorted(glob.glob(decay_files_pattern))

if not decay_files:
    print(f"No individual tree entropy files found in {decay_trends_dir}")
else:
    # Create a 3x3 grid for the 9 trees
    fig_decay, axs_decay = plt.subplots(3, 3, figsize=(9, 4.5), sharex=True, sharey=True)
    axs_flat_decay = axs_decay.flatten()
    
    # Use a distinct color palette for the trees
    decay_colors = sns.color_palette("tab10")
    
    for i, file_path in enumerate(decay_files):
        if i >= 9:
            print(f"Warning: Found more than 9 decay files, only plotting the first 9.")
            break
            
        ax = axs_flat_decay[i]
        
        try:
            # Extract tree index from filename for the title
            filename = os.path.basename(file_path)
            # Assuming filename is "bayes_trend_tree_X_..."
            tree_index_str = filename.split('_')[3]
            title = f"Tree {tree_index_str}"
            
            # Load data
            df_decay = pd.read_csv(file_path)
            time = df_decay['time_s'].values
            bayes_i = df_decay['lambda_tree_'+tree_index_str].values
            entropy = -(bayes_i*np.log2(bayes_i) + (1-bayes_i)*np.log2((1-bayes_i)))
            # Plot the entropy trend for the individual tree
            ax.plot(time, entropy, color=decay_colors[i], linewidth=2)
            ax.set_xlim(0, time[-1])
            ax.set_title(title, fontsize=14)
            ax.grid(True, linestyle='--', alpha=0.7)
            
            # Set labels only for the outer plots to avoid clutter
            if i // 3 == 2: # Bottom row
                ax.set_xlabel("Time (s)", fontsize=12)
            if i % 3 == 0: # Leftmost column
                ax.set_ylabel("Entropy", fontsize=12)

        except (IndexError, pd.errors.EmptyDataError, KeyError) as e:
            print(f"Could not process or plot {file_path}: {e}")
            ax.set_title(f"Error loading file", fontsize=12)
            ax.axis('off')

    # Hide any unused subplots if there are fewer than 9 files
    for j in range(len(decay_files), 9):
        axs_flat_decay[j].axis('off')
        
    fig_decay.suptitle("Individual Tree Belief Entropy Over Time", fontsize=suptitle_fontsize)
    fig_decay.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust for suptitle and labels
    
    output_path_decay = os.path.join(baselines_dir, "decay_trends_grid.png")
    fig_decay.savefig(output_path_decay, format='png', bbox_inches='tight', dpi=350)
    print(f"Saved individual decay plot to {output_path_decay}")

# %% [markdown]
# ## Trajectory and Entropy Animation

# %%
mode_to_animate = "mpc"
if mode_to_animate not in data_cache:
    print(f"Data for mode '{mode_to_animate}' not loaded. Skipping GIF generation.")
else:
    print(f"Generating GIF for mode: {mode_to_animate}")
    
    # --- Animation Setup ---
    data = data_cache[mode_to_animate]
    time_h, x, y, theta, entropy, lambda_h, trees = (
        data["time"], data["x"], data["y"], data["theta"],
        data["entropy"], data["lambda_h"], data["trees"]
    )

    # Animation parameters
    frame_step = 3  # Use every 3rd data point to speed up the GIF
    fade_length = 75 # Number of recent trajectory points to show with full color
    frames = []
    output_gif_path = os.path.join(baselines_dir, f"{mode_to_animate}_trajectory_entropy.gif")
    
    # --- Figure Setup ---
    # Create a figure with a 1-row, 3-column grid. The entropy plot will span columns 2 and 3.
    fig_gif = plt.figure(figsize=(18, 6.5), constrained_layout=True)
    ax_traj = plt.subplot2grid((1, 3), (0, 0))
    ax_entropy = plt.subplot2grid((1, 3), (0, 1), colspan=2)
    fig_gif.suptitle(f"N-IPP MPC: Trajectory & Entropy Evolution", fontsize=20)

    # --- Animation Loop ---
    for k in range(1, len(time_h), frame_step):
        # 1. Clear previous drawings
        ax_traj.cla()
        ax_entropy.cla()
        
        # 2. Plot Trajectory (Left Plot)
        # Set static plot elements
        ax_traj.set_aspect('equal', adjustable='box')
        # *** CHANGED: Update xlim and ylim ***
        ax_traj.set_xlim(-3.5, 9.0)
        ax_traj.set_ylim(-10.5,0.5)
        ax_traj.set_title(f"Drone Trajectory (Time: {time_h[k]:.1f}s)", fontsize=16)
        ax_traj.set_xlabel("X (m)", fontsize=14)
        ax_traj.set_ylabel("Y (m)", fontsize=14)
        ax_traj.tick_params(axis='both', labelsize=12)
        # *** CHANGED: Adjust major locator for new limits ***
        ax_traj.xaxis.set_major_locator(MultipleLocator(2))
        ax_traj.yaxis.set_major_locator(MultipleLocator(2))
        
        # *** CHANGED: Plot trees with colors updated at each step ***
        if trees is not None and trees.size > 0:
            for i in range(trees.shape[0]):
                # Use current lambda instead of final lambda
                current_lambda = lambda_h[k, i] if lambda_h.shape[1] > i else 0.5
                tree_color = custom_cmap(current_lambda)
                ax_traj.scatter(trees[i, 0], trees[i, 1], color=tree_color, s=100, marker='o', zorder=3)
        
        # Plot start position
        ax_traj.scatter(x[0], y[0], color='crimson', s=250, marker='X', label="Initial Position", zorder=4)

        # Plot fading trajectory using LineCollection
        current_x = x[:k+1]
        current_y = y[:k+1]
        points = np.array([current_x, current_y]).T.reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)
        num_segments = len(segments)
        
        base_color = to_rgba('orange')
        segment_colors = np.tile(base_color, (num_segments, 1))
        
        # Create a fade effect for the last `fade_length` segments
        fade_start_index = max(0, num_segments - fade_length)
        alphas = np.linspace(start=0.2, stop=1.0, num=num_segments - fade_start_index)
        segment_colors[fade_start_index:, 3] = alphas # Set the alpha channel
        if num_segments > fade_length:
             segment_colors[:fade_start_index, 3] = 0.2 # Older parts are very faint
                
        lc = LineCollection(segments, colors=segment_colors, linewidths=2.5)
        ax_traj.add_collection(lc)

        # Plot current drone position and orientation
        ax_traj.scatter(x[k], y[k], color='darkorange', s=120, zorder=5)
        x_orient = x[k] + 1.0 * np.cos(theta[k]) # Shortened arrow for new scale
        y_orient = y[k] + 1.0 * np.sin(theta[k])
        ax_traj.annotate("", xy=(x_orient, y_orient), xytext=(x[k], y[k]), 
                         arrowprops=dict(arrowstyle="->", color="darkorange", linewidth=2.5))
        
        # 3. Plot Entropy (Right Plot)
        # Set static plot elements
        ax_entropy.set_title("Global Entropy Decay", fontsize=16)
        ax_entropy.set_xlabel("Time (s)", fontsize=14)
        ax_entropy.set_ylabel("Entropy", fontsize=14)
        ax_entropy.tick_params(axis='both', labelsize=12)
        ax_entropy.set_xlim(0, time_h[-1])
        min_entropy, max_entropy = np.min(entropy), np.max(entropy)
        ax_entropy.set_ylim(min_entropy - 0.1 * abs(min_entropy), max_entropy + 0.1 * abs(max_entropy))
        ax_entropy.grid(True, linestyle='--', alpha=0.9)
        
        # Plot entropy curve up to current time
        ax_entropy.plot(time_h[:k+1], entropy[:k+1], color=colors[0], linewidth=2.5)
        
        # Plot a vertical line indicating current time
        ax_entropy.axvline(x=time_h[k], color='red', linestyle='--', linewidth=2, zorder=5)
        ax_entropy.scatter(time_h[k], entropy[k], color='red', s=70, zorder=6)
        
        # 4. Save frame to buffer
        buf = io.BytesIO()
        fig_gif.savefig(buf, format='png', dpi=150) # Use a moderate DPI for GIF quality
        buf.seek(0)
        frames.append(imageio.imread(buf))
        buf.close()
        
    # --- Finalize GIF ---
    plt.close(fig_gif) # Close the figure to free memory
    
    # Calculate frame duration from the data's time step
    frame_duration = (time_h[frame_step] - time_h[0]) if len(time_h) > frame_step else 0.1
    imageio.mimsave(output_gif_path, frames, duration=frame_duration)
    
    print(f"Successfully generated and saved GIF to {output_gif_path}")

# %%
# Show all generated static plots
plt.show()

: 